# Machine-learning trading analysis

An offline learning project: compare SVC and AdaBoost on the original historical CSV, select a configuration on an earlier validation period, and evaluate a later holdout. This notebook calls the same tested functions as the CLI.

Read [the README](../README.md), then choose **Restart Kernel and Run All Cells**. No API keys, broker account, or online data are required. The source symbol and price-adjustment policy are unverified; the backtest uses idealized execution and does not establish future profitability.

In [ ]:
import sys
from pathlib import Path

# Jupyter may start in this notebook's folder or in the repository root.
ROOT = Path.cwd()
if not (ROOT / "trading_analysis").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "trading_analysis").is_dir():
    raise RuntimeError("Launch Jupyter from the repository root or Starter_Code folder.")
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from trading_analysis.data import build_features, load_close
from trading_analysis.experiment import run_experiment
from trading_analysis.reporting import plot_equity

prices = load_close()
print(f"{len(prices):,} observed bars: {prices.index.min()} to {prices.index.max()}")
display(prices.head())

## 1. Inspect the input and timing

The CSV has irregular intraday observations. A four-bar moving average uses four observed closes, not four days. There is no resampling or filling of missing bars. The timestamps have no recorded timezone.

For the return ending at observation `t`, features use only closes through `t-1`. We assume the resulting position can be entered at that previous close with zero processing latency; this is an idealized backtest convention, not a verified execution price.

In [ ]:
display(prices.index.to_series().diff().value_counts().head(6).rename("gap_count"))
features = build_features(prices, fast=4, slow=100)
display(features.head())

# Changing the current and later prices cannot change this row's lagged features.
cutoff = prices.index[150]
changed = prices.copy()
changed.loc[cutoff:] *= 1.5
changed_features = build_features(changed, fast=4, slow=100)
pd.testing.assert_series_equal(
    features.loc[cutoff, ["sma_fast", "sma_slow"]],
    changed_features.loc[cutoff, ["sma_fast", "sma_slow"]],
)
print("Features at the cutoff do not depend on that close or future closes.")

## 2. Define evaluation periods before comparing results

The default experiment uses **2019 for validation** and **2020 onward for the final holdout**, with exact available timestamps recorded in metadata. Each model trains on the preceding three or six calendar months. All candidates use common evaluation timestamps, so longer SMA warmups do not change which rows are scored.

Four predeclared candidates compare the original 4/100-bar SVC, a six-month training lookback, 10/50-bar SMAs, and AdaBoost. Selection uses validation **balanced accuracy** (the mean recall across direction classes), with ties resolved in candidate order. The holdout is not used to choose parameters.

After selection, the chosen configuration and the two baseline configurations are refit using their lookback before the holdout. They then remain frozen through that holdout. This is a historical demonstration; those dates are already public and the experiment is not a prospective live test.

In [ ]:
result = run_experiment(
    validation_start="2019-01-01",
    test_start="2020-01-01",
    seed=42,
    cost_bps=5,
)
display(result.validation.round(4))
print("Selected from validation:", result.metadata["selected_candidate"])
display(pd.DataFrame(result.metadata["validation_training"]).T)
display(pd.DataFrame(result.metadata["holdout_training"]).T)

## 3. Read the holdout fairly

The baseline SVC, baseline AdaBoost, selected configuration, and an always-long benchmark all use the same holdout rows. If a baseline wins validation, its selected-model row is intentionally identical; it is not a separate improved model.

Accuracy counts correct directions. Balanced accuracy gives each observed class equal weight. Neither metric measures profits, so the table also reports compounded returns and maximum drawdown. The class reports use the true labels first, predictions second. A zero price change belongs to the up/flat class.

In [ ]:
display(result.holdout.round(4))
display(pd.DataFrame(result.reports["selected"]).T)
print("Holdout timestamps:", result.metadata["holdout_window"])
assert len({tuple(frame.index) for frame in result.predictions.values()}) == 1

## 4. Inspect positions and costs

Predictions retain their timestamp index. A `+1` position is long; `-1` is short. Because the features were already lagged, predictions multiply the matching close-to-close return without a second shift.

The example charges 5 basis points (0.05%) per unit of position turnover. Entering from cash costs one unit, reversing from long to short costs two, and the final position is liquidated. These are explicit simplified costs, not measured broker fees. Borrow fees, taxes, spread, liquidity, margin, and latency are omitted.

In [ ]:
display(result.predictions["selected"].head(8))
figure = plot_equity(result)
plt.show()

## 5. Interpret the result and explain the fixes

The checked-in [evaluation report](../reports/example/README.md) shows the default experiment's measured results. The winning validation balanced accuracy is only slightly above 0.5, so an attractive holdout return does not establish a robust predictive edge. Changing dates, features, or costs after seeing holdout results turns that period into exploratory data.

The original SVM trained on scaled features but predicted on raw features; the shared pipeline fixes that mismatch. The original SVM report reversed its arguments; prediction tables also lost their timestamp index, and the AdaBoost backtest reused SVM predictions. Tests now cover each behavior.

Run the CLI from the repository root to save tables, per-bar predictions, source hashes, class reports, and figures:

```bash
python -m trading_analysis
```

Use [WALKTHROUGH.md](../docs/WALKTHROUGH.md) to trace the code, [METHOD.md](../docs/METHOD.md) to check equations and assumptions, and [DATA.md](../docs/DATA.md) to review input limitations.